# Animal economics

Steady-state revenue per animal per day, net of the 1 wheat/day feed cost. Break-even day at base prices. Care bonus: `+1 pending` on each fed-and-cared day, paid out on the next scheduled production if fed.

Feed cost = 1 wheat per animal per day. If we sell wheat at base $25, that is the opportunity cost. If we grow wheat on-farm, the marginal cost is essentially the tile-day of wheat production per animal per day.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kaggriculture.env.constants import ANIMALS, MARKET_PARAMS

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def cumulative_net(animal: str, days: int, cared: bool = False, feed_price: int = 25) -> list[int]:
    """Cumulative net revenue at base prices, net of wheat feed. `cared` doubles yield on each production tick."""
    a = ANIMALS[animal]
    product = a["product"]
    price = MARKET_PARAMS[product]["base"]
    cost = a["cost"]  # buy price
    cum = -cost  # start with capex sunk
    pending_care = 0
    trace = []
    for day in range(days + 1):
        days_since = day - a["first_yield_day"]
        if days_since >= 0 and days_since % a["interval"] == 0:
            units = 1 + pending_care
            cum += units * price
            pending_care = 0
        # feed cost every day the animal is alive
        if day > 0:
            cum -= feed_price
        # bank care bonus for tomorrow
        if cared:
            pending_care += 1
        trace.append(cum)
    return trace

In [ ]:
horizon = 30
days = np.arange(horizon + 1)
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"GOOSE": "#1f77b4", "COW": "#2ca02c", "SHEEP": "#d62728"}
for animal in ANIMALS:
    ax.plot(
        days,
        cumulative_net(animal, horizon, cared=False),
        color=colors[animal],
        label=f"{animal} (no care)",
    )
    ax.plot(
        days,
        cumulative_net(animal, horizon, cared=True),
        color=colors[animal],
        linestyle="--",
        label=f"{animal} (cared)",
    )

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("day")
ax.set_ylabel("cumulative net revenue ($, base prices, wheat cost $25)")
ax.set_title("Animal cumulative net revenue over 30 days")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "animal-cumnet.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for animal, a in ANIMALS.items():
    product = a["product"]
    price = MARKET_PARAMS[product]["base"]
    prod_per_day = 1 / a["interval"]
    gross = prod_per_day * price
    net_wheat_market = gross - 25
    net_wheat_free = gross - 0
    cost = a["cost"]
    breakeven_market = cost / max(1e-6, net_wheat_market)
    breakeven_free = cost / max(1e-6, net_wheat_free)
    rows.append(
        {
            "animal": animal,
            "buy_cost": cost,
            "first_yield_day": a["first_yield_day"],
            "interval": a["interval"],
            "steady_units/day": round(prod_per_day, 3),
            "gross_$/day": round(gross, 2),
            "net_$/day_wheat25": round(net_wheat_market, 2),
            "net_$/day_free_wheat": round(net_wheat_free, 2),
            "days_to_breakeven_wheat25": round(breakeven_market + a["first_yield_day"], 1),
            "days_to_breakeven_free_wheat": round(breakeven_free + a["first_yield_day"], 1),
        }
    )

pd.DataFrame(rows).set_index("animal")

## Takeaways

- Goose is the fastest ROI by every metric: cheap ($300), first yield on day 4, produces every day. At base prices with market-cost wheat ($25/day) net revenue is $25/day, break-even around day 16. With farm-grown wheat, break-even is around day 10 and net is $50/day.
- Cow has the best gross ($80/day at base) but 8-day setup lag, higher capex ($400), and every-two-days interval halves the daily rate. Break-even is around day 14 with market wheat, day 13 with free wheat.
- Sheep has the highest single-yield price ($200) but the slowest interval (every 3 days), giving a modest $66/day gross. High variance since wool crashes hard on glut (see notebook 01).
- Care bonus is small but strictly additive: over a 30-day season a cared goose earns ~30 extra units versus uncared.
- The dominant driver of animal ROI is the wheat cost model. Growing wheat on-farm changes goose net revenue by 2x.